In [ ]:
import cv2 as cv
import numpy as np
from ultralytics import YOLO

# YOLOv8 모델 로드
model = YOLO('D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt')  # 훈련된 모델 파일 경로로 변경해주세요
# 이미지 읽기
img1 = cv.imread('D:/LHJ-Capstone Wire detection/wireod/test (5).jpg')  # 첫 번째 이미지
img2 = cv.imread('D:/LHJ-Capstone Wire detection/wireod/test (6).jpg')  # 두 번째 이미지

# YOLOv8로 wire 객체 검출
results1 = model(img1)
results2 = model(img2)

# wire 객체의 바운딩 박스 추출 (cls 수정)
def get_wire_boxes(results):
    boxes = []
    for r in results:
        for box in r.boxes:
            # YOLOv8에서 클래스 ID 확인, 0이 wire 클래스인지 확인
            if int(box.cls) == 0:  # wire 클래스가 0번째 인덱스라고 가정
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                boxes.append([x1, y1, x2 - x1, y2 - y1])
    return boxes

boxes1 = get_wire_boxes(results1)
boxes2 = get_wire_boxes(results2)

# IOU (Intersection over Union) 계산 함수
def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    
    box1_area = w1 * h1
    box2_area = w2 * h2
    
    iou = inter_area / float(box1_area + box2_area - inter_area)
    return iou

# IOU와 좌표 차이를 고려한 변화 분석 함수
def analyze_changes(boxes1, boxes2, iou_threshold=0.1, coord_diff_threshold=10):
    changes = []
    for box1 in boxes1:
        matched = False
        for box2 in boxes2:
            iou = calculate_iou(box1, box2)
            coord_diff = np.sum(np.abs(np.array(box1) - np.array(box2)))
            if iou > iou_threshold or coord_diff < coord_diff_threshold:
                matched = True
                break
        if not matched:
            changes.append(('removed', box1))
    
    for box2 in boxes2:
        matched = False
        for box1 in boxes1:
            iou = calculate_iou(box1, box2)
            coord_diff = np.sum(np.abs(np.array(box1) - np.array(box2)))
            if iou > iou_threshold or coord_diff < coord_diff_threshold:
                matched = True
                break
        if not matched:
            changes.append(('added', box2))
    
    return changes

# 변경된 함수로 변화 분석 실행
changes = analyze_changes(boxes1, boxes2)

# 결과 시각화
result_img = img2.copy()
for change_type, box in changes:
    color = (0, 255, 0) if change_type == 'added' else (0, 0, 255)
    x, y, w, h = box
    cv.rectangle(result_img, (x, y), (x+w, y+h), color, 2)
    cv.putText(result_img, change_type, (x, y-10), cv.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

# 결과 출력
print(f"변화 수: {len(changes)}")
for change_type, box in changes:
    print(f"{change_type}: {box}")

cv.imshow('Wire Changes', result_img)
cv.waitKey(0)
cv.destroyAllWindows()


In [ ]:
import cv2 as cv
import numpy as np
from ultralytics import YOLO

# YOLOv8 모델 로드
model = YOLO('D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt')  # 훈련된 모델 파일 경로로 변경해주세요
# 이미지 읽기
img1 = cv.imread('D:/LHJ-Capstone Wire detection/wireod/test (5).jpg')  # 첫 번째 이미지
img2 = cv.imread('D:/LHJ-Capstone Wire detection/wireod/test (6).jpg')  # 두 번째 이미지

# YOLOv8로 wire 객체 검출
results1 = model(img1)
results2 = model(img2)

# wire 객체의 바운딩 박스 추출 (cls 수정)
def get_wire_boxes(results):
    boxes = []
    for r in results:
        for box in r.boxes:
            # YOLOv8에서 클래스 ID 확인, 0이 wire 클래스인지 확인
            if int(box.cls) == 0:  # wire 클래스가 0번째 인덱스라고 가정
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                boxes.append([x1, y1, x2 - x1, y2 - y1])
    return boxes

boxes1 = get_wire_boxes(results1)
boxes2 = get_wire_boxes(results2)

# IOU (Intersection over Union) 계산 함수
def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    
    box1_area = w1 * h1
    box2_area = w2 * h2
    
    iou = inter_area / float(box1_area + box2_area - inter_area)
    return iou

# IOU와 좌표 차이를 고려한 변화 분석 함수 (박스 수 비교 추가)
def analyze_changes(boxes1, boxes2, iou_threshold=0.1, coord_diff_threshold=10):
    changes = []
    
    # 박스 수가 다른 경우 무조건 변화로 간주
    if len(boxes1) != len(boxes2):
        changes.append(('different count', (len(boxes1), len(boxes2))))
    
    # 기존 박스 매칭 로직
    for box1 in boxes1:
        matched = False
        for box2 in boxes2:
            iou = calculate_iou(box1, box2)
            coord_diff = np.sum(np.abs(np.array(box1) - np.array(box2)))
            if iou > iou_threshold or coord_diff < coord_diff_threshold:
                matched = True
                break
        if not matched:
            changes.append(('removed', box1))
    
    for box2 in boxes2:
        matched = False
        for box1 in boxes1:
            iou = calculate_iou(box1, box2)
            coord_diff = np.sum(np.abs(np.array(box1) - np.array(box2)))
            if iou > iou_threshold or coord_diff < coord_diff_threshold:
                matched = True
                break
        if not matched:
            changes.append(('added', box2))
    
    return changes

# 변경된 함수로 변화 분석 실행
changes = analyze_changes(boxes1, boxes2)

# 결과 시각화
result_img = img2.copy()
for change_type, change_info in changes:
    if change_type == 'different count':
        print(f"Wire count difference: Image 1 has {change_info[0]} wires, Image 2 has {change_info[1]} wires")
    else:
        color = (0, 255, 0) if change_type == 'added' else (0, 0, 255)
        x, y, w, h = change_info
        cv.rectangle(result_img, (x, y), (x+w, y+h), color, 2)
        cv.putText(result_img, change_type, (x, y-10), cv.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

# 결과 출력
print(f"변화 수: {len(changes)}")
for change_type, change_info in changes:
    if change_type == 'different count':
        print(f"Wire count difference: Image 1 has {change_info[0]} wires, Image 2 has {change_info[1]} wires")
    else:
        print(f"{change_type}: {change_info}")

cv.imshow('Wire Changes', result_img)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv
import numpy as np
from ultralytics import YOLO

# YOLOv8 모델 로드
model = YOLO('D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt')  # 훈련된 모델 파일 경로로 변경해주세요
# 이미지 읽기
img1 = cv.imread('D:/LHJ-Capstone Wire detection/wireod/test (5).jpg')  # 첫 번째 이미지
img2 = cv.imread('D:/LHJ-Capstone Wire detection/wireod/test (6).jpg')  # 두 번째 이미지

# YOLOv8로 wire 객체 검출
results1 = model(img1)
results2 = model(img2)

# wire 객체의 바운딩 박스 추출 (cls 수정)
def get_wire_boxes(results):
    boxes = []
    for r in results:
        for box in r.boxes:
            # YOLOv8에서 클래스 ID 확인, 0이 wire 클래스인지 확인
            if int(box.cls) == 0:  # wire 클래스가 0번째 인덱스라고 가정
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                boxes.append([x1, y1, x2 - x1, y2 - y1])
    return boxes

boxes1 = get_wire_boxes(results1)
boxes2 = get_wire_boxes(results2)

# IOU (Intersection over Union) 계산 함수
def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    
    box1_area = w1 * h1
    box2_area = w2 * h2
    
    iou = inter_area / float(box1_area + box2_area - inter_area)
    return iou

# 바운딩 박스 비교 및 변화 감지
def analyze_changes(boxes1, boxes2, iou_threshold=0.1, coord_diff_threshold=10):
    changes = []
    
    # 박스 수가 다를 때 변화 추가
    if len(boxes1) != len(boxes2):
        if len(boxes1) > len(boxes2):
            for box1 in boxes1:
                changes.append(('removed', box1))
        else:
            for box2 in boxes2:
                changes.append(('added', box2))
    
    # 기존 박스 매칭 로직
    for box1 in boxes1:
        matched = False
        for box2 in boxes2:
            iou = calculate_iou(box1, box2)
            coord_diff = np.sum(np.abs(np.array(box1) - np.array(box2)))
            if iou > iou_threshold or coord_diff < coord_diff_threshold:
                matched = True
                break
        if not matched:
            changes.append(('removed', box1))
    
    for box2 in boxes2:
        matched = False
        for box1 in boxes1:
            iou = calculate_iou(box1, box2)
            coord_diff = np.sum(np.abs(np.array(box1) - np.array(box2)))
            if iou > iou_threshold or coord_diff < coord_diff_threshold:
                matched = True
                break
        if not matched:
            changes.append(('added', box2))
    
    return changes

# 변경된 함수로 변화 분석 실행
changes = analyze_changes(boxes1, boxes2)

# 결과 시각화
result_img = img2.copy()
for change_type, change_info in changes:
    color = (0, 255, 0) if change_type == 'added' else (0, 0, 255)
    x, y, w, h = change_info
    cv.rectangle(result_img, (x, y), (x+w, y+h), color, 2)
    cv.putText(result_img, change_type, (x, y-10), cv.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

# 결과 출력
print(f"변화 수: {len(changes)}")
for change_type, change_info in changes:
    print(f"{change_type}: {change_info}")

cv.imshow('Wire Changes', result_img)
cv.waitKey(0)
cv.destroyAllWindows()


In [ ]:
import cv2 as cv
import numpy as np
from ultralytics import YOLO

# YOLOv8 모델 로드
model = YOLO('D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt')  # 훈련된 모델 파일 경로로 변경해주세요
# 이미지 읽기
img1 = cv.imread('D:/LHJ-Capstone Wire detection/wireod/test (5).jpg')  # 첫 번째 이미지
img2 = cv.imread('D:/LHJ-Capstone Wire detection/wireod/test (6).jpg')  # 두 번째 이미지

# YOLOv8로 wire 객체 검출
results1 = model(img1)
results2 = model(img2)

def get_wire_boxes(results):
    boxes = []
    for r in results:
        for box in r.boxes:
            if int(box.cls) == 0:  # wire 클래스가 0번째 인덱스라고 가정
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                boxes.append([x1, y1, x2 - x1, y2 - y1])
    return boxes

boxes1 = get_wire_boxes(results1)
boxes2 = get_wire_boxes(results2)

def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    
    box1_area = w1 * h1
    box2_area = w2 * h2
    
    iou = inter_area / float(box1_area + box2_area - inter_area)
    return iou

def analyze_changes(boxes1, boxes2, iou_threshold=0.1, coord_diff_threshold=10):
    changes = []
    
    if len(boxes1) != len(boxes2):
        changes.append(('count_change', (len(boxes1), len(boxes2))))
    
    matched_boxes2 = set()
    
    for i, box1 in enumerate(boxes1):
        matched = False
        for j, box2 in enumerate(boxes2):
            iou = calculate_iou(box1, box2)
            coord_diff = np.sum(np.abs(np.array(box1) - np.array(box2)))
            if iou > iou_threshold or coord_diff < coord_diff_threshold:
                matched = True
                matched_boxes2.add(j)
                break
        if not matched:
            changes.append(('removed', box1))
    
    for j, box2 in enumerate(boxes2):
        if j not in matched_boxes2:
            changes.append(('added', box2))
    
    return changes

changes = analyze_changes(boxes1, boxes2)

# 결과 시각화
result_img = img2.copy()

# 모든 와이어 박스 표시
for box in boxes2:
    x, y, w, h = box
    cv.rectangle(result_img, (x, y), (x+w, y+h), (255, 255, 0), 2)  # 노란색으로 모든 와이어 표시

# 변화 표시
for change_type, change_info in changes:
    if change_type == 'count_change':
        print(f"Wire count changed: Image 1 has {change_info[0]} wires, Image 2 has {change_info[1]} wires")
    elif change_type == 'added':
        x, y, w, h = change_info
        cv.rectangle(result_img, (x, y), (x+w, y+h), (0, 255, 0), 3)  # 초록색으로 새로 추가된 와이어 강조
        cv.putText(result_img, 'New', (x, y-10), cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    elif change_type == 'removed':
        print(f"Removed wire: {change_info}")

# 결과 출력
print(f"변화 수: {len(changes)}")
for change_type, change_info in changes:
    print(f"{change_type}: {change_info}")

cv.imshow('Wire Changes', result_img)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
# YOLOv8 결과 시각화
def visualize_detection(img, results):
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls)
            score = box.conf[0]  # 신뢰도 점수
            label = f"Class {cls} ({score:.2f})"
            color = (0, 255, 0)  # 초록색
            cv.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv.putText(img, label, (x1, y1 - 10), cv.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv.imshow('YOLO Detection', img)
    cv.waitKey(0)
    cv.destroyAllWindows()

# 각 이미지에 대한 탐지 시각화
visualize_detection(img1.copy(), results1)
visualize_detection(img2.copy(), results2)


In [ ]:
# YOLOv8 결과에서 클래스 인덱스 출력
def print_classes(results):
    for r in results:
        for box in r.boxes:
            print(f"Detected class index: {int(box.cls)} with confidence {box.conf[0]:.2f}")

print("Image 1 classes:")
print_classes(results1)

print("Image 2 classes:")
print_classes(results2)


In [ ]:
# 탐지된 바운딩 박스 출력
def print_boxes(boxes, image_name):
    print(f"Boxes in {image_name}:")
    for box in boxes:
        print(box)

print_boxes(boxes1, "Image 1")
print_boxes(boxes2, "Image 2")